[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/19-reading-a-model-card.ipynb)

# Module 19: Reading a Model Card

Companion notebook to [Module 19](https://transformers-and-math.drmhse.com/modules/19-reading-a-model-card) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import json, urllib.request

## Part 1 — Load real configuration files (section 19.2)
Configs are small JSON files on the Hugging Face Hub. Some models require accepting a licence before download; the fallbacks below have the same fields.

In [ ]:
FALLBACK = {
    'Qwen/Qwen2.5-7B': dict(vocab_size=152064, hidden_size=3584, num_hidden_layers=28, num_attention_heads=28, num_key_value_heads=4, intermediate_size=18944, hidden_act='silu', tie_word_embeddings=False, torch_dtype='bfloat16'),
    'openai-community/gpt2': dict(vocab_size=50257, n_embd=768, n_layer=12, n_head=12, n_inner=None, activation_function='gelu_new', tie_word_embeddings=True),
}

def load(repo):
    try:
        with urllib.request.urlopen(f'https://huggingface.co/{repo}/resolve/main/config.json', timeout=10) as r:
            return json.load(r)
    except Exception:
        return FALLBACK[repo]

def normalize(c):
    """GPT-2 uses older field names; map them to the LLaMA-style names."""
    if 'n_embd' in c:
        return dict(vocab_size=c['vocab_size'], hidden_size=c['n_embd'], num_hidden_layers=c['n_layer'],
                    num_attention_heads=c['n_head'], num_key_value_heads=c['n_head'],
                    intermediate_size=c.get('n_inner') or 4 * c['n_embd'], hidden_act='gelu', tie_word_embeddings=True, torch_dtype='float32')
    return c

configs = {repo: normalize(load(repo)) for repo in FALLBACK}
for repo, c in configs.items():
    print(repo, {k: c[k] for k in ('hidden_size', 'num_hidden_layers', 'num_attention_heads', 'num_key_value_heads', 'intermediate_size')})

## Part 2 — Parameters, KV cache and compute from a config (section 19.3)

In [ ]:
def estimate(c, context=8192):
    d, L, hq = c['hidden_size'], c['num_hidden_layers'], c['num_attention_heads']
    hkv, dff, V = c.get('num_key_value_heads', hq), c['intermediate_size'], c['vocab_size']
    dh = d // hq
    gated = c.get('hidden_act') == 'silu'
    attn = 2 * d * d + 2 * d * hkv * dh
    ffn = (3 if gated else 2) * d * dff
    total = L * (attn + ffn) + V * d * (1 if c.get('tie_word_embeddings') else 2)
    nbytes = {'float32': 4}.get(c.get('torch_dtype'), 2)
    kv = 2 * L * hkv * dh * context * nbytes
    return total, kv, 2 * total

for repo, c in configs.items():
    total, kv, flops = estimate(c)
    print(f'{repo:24}  ≈{total / 1e9:5.2f}B parameters   KV cache at 8,192 tokens {kv / 2 ** 30:5.2f} GiB   ≈{flops / 1e9:5.1f} GFLOP per token')

## Your turn
1. Load a config for a model you are interested in (for example `HuggingFaceTB/SmolLM2-1.7B`) and estimate its size. Compare with its model card.
2. Change `num_key_value_heads` to equal `num_attention_heads` and see how much the KV cache grows, and how little the parameter count does.